# Colab ESG 챗봇 통합 테스트
이 노트북 하나에 전처리 모듈과 Seq2Seq 코드가 포함되어 있습니다. 별도 .py 파일은 필요하지 않습니다.
1. Colab에 이 노트북을 업로드하고, 필요하면 런타임 설정에서 GPU를 선택하세요.
2. 아래 설정에서 MODE="train"을 선택한 후 순서대로 실행합니다. 요청 시 두 CSV를 업로드하세요.
3. 데이터 분석 그래프 → 학습 그래프 → 모델 재로드 → 질문 테스트가 진행됩니다.
4. 마지막 다운로드 셀에서 가중치와 설정을 보관하세요. MODE="chat"은 두 모델 파일을 업로드하여 재학습 없이 대화합니다.
Colab 런타임 저장소는 임시입니다. [Colab 안내](https://research.google.com/colaboratory/faq.html).
라이브러리는 런타임에 설치된 것을 사용하며 자동 설치하지 않습니다. 로컬 실행은 esg 환경만 사용하세요.

In [ ]:
import os
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass
if not IN_COLAB and Path(sys.prefix).name.lower() != "esg":
    raise RuntimeError("로컬에서는 esg 환경을 사용하세요.")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import json
import numpy as np
import tensorflow as tf
tf.keras.utils.set_random_seed(1234)
print("TensorFlow:", tf.__version__, "GPU:", tf.config.list_physical_devices("GPU"))

MODE = "train"  # 기존 모델 테스트: "chat"
EPOCHS = 20
BATCH_SIZE = 32  # 메모리 부족 시 더 줄이세요.
UNITS = 128
EMBEDDING_DIM = 64
MAX_LENGTH = 256
MAX_SAMPLES = None  # 첫 실행 점검: 128 및 EPOCHS=1
BASE = Path("/content/esg_chatbot") if IN_COLAB else Path.cwd() / "colab_test_output"
BASE.mkdir(parents=True, exist_ok=True)
DATA_PATHS = [BASE / "ChatbotData.csv", BASE / "ESG_QnA_dataset_10000.csv"]
PREPARED_DIR = None
PREPROCESS_ROOT = BASE / "data_in"
RUN_DIR = BASE / "data_out" / "seq2seq_char"
WEIGHTS, CONFIG = RUN_DIR / "best.weights.h5", RUN_DIR / "config.json"
assert MODE in {"train", "chat"}

if IN_COLAB:
    from google.colab import files
    if MODE == "train" and any(not p.exists() for p in DATA_PATHS):
        print("ChatbotData.csv와 ESG_QnA_dataset_10000.csv를 업로드하세요.")
        uploaded = files.upload()
        for target in DATA_PATHS:
            if target.name in uploaded:
                target.write_bytes(uploaded[target.name])
        if any(not p.exists() for p in DATA_PATHS):
            raise FileNotFoundError("두 CSV가 모두 필요합니다. 이 셀을 다시 실행하세요.")
    if MODE == "chat" and not list(RUN_DIR.glob("*/best.weights.h5")):
        print("같은 학습에서 저장된 best.weights.h5와 config.json을 업로드하세요.")
        uploaded = files.upload()
        if not {"best.weights.h5", "config.json"}.issubset(uploaded):
            raise FileNotFoundError("두 모델 파일이 모두 필요합니다.")
        from datetime import datetime
        imported = RUN_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        imported.mkdir(parents=True, exist_ok=False)
        for name in ["best.weights.h5", "config.json"]:
            (imported / name).write_bytes(uploaded[name])

In [ ]:
# 전처리 모듈 구현 (preprocessing.py와 동일)
"""CSV analysis and character preprocessing, independent of TensorFlow."""
import argparse
import csv
import hashlib
import json
import re
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np

PAD, SOS, EOS, UNK = 0, 1, 2, 3


def encode(text, vocabulary, length):
    ids = [vocabulary.get(char, UNK) for char in text.strip()[:length]]
    return ids + [PAD] * (length - len(ids))


def read_data(paths):
    rows, seen, sources = [], set(), []
    for path in map(Path, paths):
        counts = dict(file=path.name, sha256=hashlib.sha256(path.read_bytes()).hexdigest(), rows=0, empty=0, duplicates=0, added=0)
        with path.open(encoding='utf-8-sig', newline='') as stream:
            reader = csv.DictReader(stream)
            if not {'Q', 'A'}.issubset(reader.fieldnames or []):
                raise ValueError(f'{path.name}: Q, A 열이 필요합니다.')
            for row in reader:
                counts['rows'] += 1
                pair = ((row.get('Q') or '').strip(), (row.get('A') or '').strip())
                if not all(pair):
                    counts['empty'] += 1
                elif pair in seen:
                    counts['duplicates'] += 1
                else:
                    seen.add(pair)
                    rows.append(dict(Q=pair[0], A=pair[1], source=path.name))
                    counts['added'] += 1
        sources.append(counts)
    if len(rows) < 10:
        raise ValueError('최소 10개 유효 질문·답변이 필요합니다.')
    return rows, sources


def analyze(rows, sources, max_length):
    report = dict(data_sources=sources, count=len(rows), max_length=max_length)
    for field in ['Q', 'A']:
        lengths = np.array([len(row[field]) for row in rows])
        limit = max_length if field == 'Q' else max_length - 1
        report[field] = dict(min=int(lengths.min()), mean=float(lengths.mean()),
                             p50=float(np.percentile(lengths, 50)), p90=float(np.percentile(lengths, 90)),
                             p95=float(np.percentile(lengths, 95)), max=int(lengths.max()),
                             truncated=int((lengths > limit).sum()), limit=limit)
    answers = defaultdict(set)
    for row in rows:
        answers[row['Q']].add(row['A'])
    report['multiple_answer_questions'] = sum(len(values) > 1 for values in answers.values())
    # Surface word frequency, not a morphological or semantic analysis.
    report['top_words'] = {}
    for field in ['Q', 'A']:
        counter = Counter(word.lower() for row in rows for word in re.findall(r'[가-힣A-Za-z0-9]+', row[field]) if len(word) > 1)
        report['top_words'][field] = counter.most_common(30)
    return report


def plot_analysis(rows, output):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for ax, field, title in zip(axes, ['Q', 'A'], ['Question length', 'Answer length']):
        for source in sorted({row['source'] for row in rows}):
            ax.hist([len(row[field]) for row in rows if row['source'] == source], bins=40, alpha=0.55, label=source)
        ax.set(title=title, xlabel='Characters', ylabel='Number of records')
        ax.legend(fontsize=8)
    fig.savefig(Path(output) / 'length_distribution.png', dpi=140)
    plt.close(fig)


def prepare(paths, output_root, max_length=256, max_samples=None, seed=1234):
    if max_length < 2 or (max_samples is not None and max_samples < 10):
        raise ValueError('max_length >= 2, max_samples >= 10 이어야 합니다.')
    rows, sources = read_data(paths)
    report = analyze(rows, sources, max_length)
    rng = np.random.default_rng(seed)
    # Keep identical (truncated) model inputs in one split to avoid validation leakage.
    groups = defaultdict(list)
    for row in rows:
        groups[row['Q'][:max_length]].append(row)
    keys = list(groups)
    rng.shuffle(keys)
    selected = []
    for key in keys:
        selected.extend(groups[key])
        if max_samples is not None and len(selected) >= max_samples:
            break  # retain whole question groups; sample limit is approximate
    grouped = defaultdict(list)
    for row in selected:
        grouped[row['Q'][:max_length]].append(row)
    if len(grouped) < 2:
        raise ValueError('학습·검증 분리를 위해 서로 다른 질문이 최소 2개 필요합니다.')
    cutoff = max(1, min(len(grouped) - 1, int(len(grouped) * 0.9)))
    train = [row for group in list(grouped.values())[:cutoff] for row in group]
    valid = [row for group in list(grouped.values())[cutoff:] for row in group]
    selected = train + valid
    chars = sorted(set(''.join(r['Q'][:max_length] + r['A'][:max_length - 1] for r in train)))
    vocabulary = {char: i + 4 for i, char in enumerate(chars)}
    x = np.asarray([encode(r['Q'], vocabulary, max_length) for r in selected], dtype='int32')
    targets = [[vocabulary.get(c, UNK) for c in r['A'][:max_length - 1]] + [EOS] for r in selected]
    y = np.asarray([t + [PAD] * (max_length - len(t)) for t in targets], dtype='int32')
    decoder_inputs = np.concatenate([np.full((len(y), 1), SOS, dtype='int32'), y[:, :-1]], axis=1)
    output = Path(output_root) / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    output.mkdir(parents=True, exist_ok=False)
    config = dict(format_version=1, vocabulary=vocabulary, max_length=max_length, split=len(train),
                  sample_count=len(selected), data_sources=sources, seed=seed)
    np.savez_compressed(output / 'training_data.npz', x=x, y=y, decoder_inputs=decoder_inputs)
    (output / 'preprocess_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    report['training_selection'] = dict(train=len(train), validation=len(valid), vocabulary_size=len(vocabulary)+4,
        truncated_questions=sum(len(r['Q']) > max_length for r in selected),
        truncated_answers=sum(len(r['A']) >= max_length for r in selected))
    (output / 'analysis.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    with (output / 'split_records.csv').open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['Q', 'A', 'source', 'split'])
        writer.writeheader()
        writer.writerows(dict(row, split='train' if i < len(train) else 'validation') for i, row in enumerate(selected))
    plot_analysis(rows, output)
    for source in sources:
        print(f"{source['file']}: {source['rows']:,}행 / 사용 {source['added']:,} / 중복 {source['duplicates']} / 빈 행 {source['empty']}")
    print(f"통합 {len(rows):,} / 학습 {len(train):,} / 검증 {len(valid):,}")
    print(f"전체 데이터 중 길이 초과: 질문 {report['Q']['truncated']:,} / 답변 {report['A']['truncated']:,}")
    print('저장:', output)
    return output, report




## 1. 전처리 모듈 호출 및 분석 결과
데이터 통합·정제·어휘 생성은 preprocessing.py가 담당하며 그래프와 통계는 이 노트북에 표시합니다.

In [ ]:
if MODE == "train":
    from datetime import datetime
    if PREPARED_DIR is None:
        PREPARED_DIR, _ = prepare(DATA_PATHS, PREPROCESS_ROOT, MAX_LENGTH, MAX_SAMPLES)
    PREPARED_DIR = Path(PREPARED_DIR)
    config = json.loads((PREPARED_DIR / "preprocess_config.json").read_text(encoding="utf-8"))
    with np.load(PREPARED_DIR / "training_data.npz", allow_pickle=False) as arrays:
        x, y, decoder_inputs = arrays["x"], arrays["y"], arrays["decoder_inputs"]
    split = config["split"]
    if not (x.shape == y.shape == decoder_inputs.shape == (config["sample_count"], config["max_length"]) and 0 < split < len(x)):
        raise ValueError("전처리 배열과 설정이 일치하지 않습니다. 전처리를 다시 실행하세요.")
    config.update(units=UNITS, embedding_dim=EMBEDDING_DIM, prepared_dir=str(PREPARED_DIR))
    vocabulary = config["vocabulary"]
    RUN_DIR = RUN_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    WEIGHTS, CONFIG = RUN_DIR / "best.weights.h5", RUN_DIR / "config.json"
    CONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    print("전처리 결과:", PREPARED_DIR)
    print(f"학습 {split:,} / 검증 {len(x)-split:,}")
else:
    # 특정 학습 결과를 선택하려면 아래 자동 선택 대신 RUN_DIR을 직접 지정하세요.
    saved = sorted(p.parent for p in RUN_DIR.glob("*/best.weights.h5") if (p.parent / "config.json").exists())
    if not saved:
        raise FileNotFoundError("저장 모델이 없습니다. 먼저 MODE='train'으로 학습하세요.")
    RUN_DIR = saved[-1]
    WEIGHTS, CONFIG = RUN_DIR / "best.weights.h5", RUN_DIR / "config.json"
    config = json.loads(CONFIG.read_text(encoding="utf-8"))
    vocabulary = config["vocabulary"]
print("모델 폴더:", RUN_DIR)
if MODE == "train":
    import pandas as pd
    from IPython.display import display, Image
    report = json.loads((PREPARED_DIR / "analysis.json").read_text(encoding="utf-8"))
    display(pd.DataFrame(report["data_sources"]).drop(columns="sha256", errors="ignore"))
    display(pd.DataFrame({field: report[field] for field in ["Q", "A"]}).T)
    print("동일 질문에 여러 답변이 있는 질문 수:", report["multiple_answer_questions"])
    display(Image(filename=str(PREPARED_DIR / "length_distribution.png")))
    for field in ["Q", "A"]:
        print(field, "빈도 상위 단어 (표면형 집계, 형태소 분석 아님)")
        display(pd.DataFrame(report["top_words"][field], columns=["단어", "빈도"]))

## 2. 모델 구성
Encoder의 상태를 Decoder 초기 상태로 전달합니다. 학습과 추론이 동일한 레이어를 공유합니다.

In [ ]:
def build_models(config):
    size = len(config["vocabulary"]) + 4
    units, dim = config["units"], config["embedding_dim"]
    encoder_tokens = tf.keras.Input(shape=(None,), dtype="int32", name="encoder_tokens")
    enc_embedding = tf.keras.layers.Embedding(size, dim, mask_zero=True)
    enc_gru = tf.keras.layers.GRU(units, return_state=True)
    _, state = enc_gru(enc_embedding(encoder_tokens))
    decoder_tokens = tf.keras.Input(shape=(None,), dtype="int32", name="decoder_tokens")
    dec_embedding = tf.keras.layers.Embedding(size, dim, mask_zero=True)
    dec_gru = tf.keras.layers.GRU(units, return_sequences=True, return_state=True)
    projection = tf.keras.layers.Dense(size)
    sequence, _ = dec_gru(dec_embedding(decoder_tokens), initial_state=state)
    training = tf.keras.Model([encoder_tokens, decoder_tokens], projection(sequence))
    encoder = tf.keras.Model(encoder_tokens, state)
    previous_state = tf.keras.Input(shape=(units,), name="previous_state")
    sequence, next_state = dec_gru(dec_embedding(decoder_tokens), initial_state=previous_state)
    decoder = tf.keras.Model([decoder_tokens, previous_state], [projection(sequence), next_state])
    return training, encoder, decoder

def masked_loss(actual, predicted):
    loss = tf.keras.losses.sparse_categorical_crossentropy(actual, predicted, from_logits=True)
    mask = tf.cast(tf.not_equal(actual, PAD), loss.dtype)
    return tf.math.divide_no_nan(tf.reduce_sum(loss * mask), tf.reduce_sum(mask))

class MaskedTokenAccuracy(tf.keras.metrics.SparseCategoricalAccuracy):
    """전체 비-PAD 정답 토큰에 대한 정확도 (EOS 포함)."""
    def __init__(self, name="accuracy", **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        weights = tf.cast(tf.not_equal(y_true, PAD), self.dtype)
        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, self.dtype)
            if sample_weight.shape.rank == 1:
                sample_weight = tf.expand_dims(sample_weight, -1)
            weights *= sample_weight
        return super().update_state(y_true, y_pred, sample_weight=weights)


model, encoder, decoder = build_models(config)
model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss=masked_loss, metrics=[MaskedTokenAccuracy()])
model.summary()

## 3. 학습 및 저장
검증 손실이 가장 낮은 가중치를 저장합니다. chat 모드에서는 이 단계를 건너뜁니다.
`loss`, `accuracy`, `val_loss`, `val_accuracy`를 함께 표시하고 history.json에 저장합니다. 정확도는 PAD를 제외하고 EOS를 포함한 문자 토큰 기준이며, 답변의 사실성 점수가 아닙니다. 최적 모델 선택 기준은 val_loss입니다.


In [ ]:
if MODE == "train":
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(WEIGHTS), monitor="val_loss", save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        tf.keras.callbacks.TerminateOnNaN(),
    ]
    history = model.fit(
        [x[:split], decoder_inputs[:split]], y[:split],
        validation_data=([x[split:], decoder_inputs[split:]], y[split:]),
        batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks,
    )
    (RUN_DIR / "history.json").write_text(json.dumps(history.history, indent=2), encoding="utf-8")
    print("저장:", WEIGHTS)
if MODE == "train":
    import matplotlib.pyplot as plt
    from IPython.display import display, Image
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        epochs = range(1, len(history.history[metric]) + 1)
        ax.plot(epochs, history.history[metric], label="train")
        ax.plot(epochs, history.history["val_" + metric], label="validation")
        ax.set(xlabel="Epoch", ylabel=metric, title=metric)
        ax.legend()
    fig.savefig(RUN_DIR / "training_curves.png", dpi=140)
    plt.close(fig)
    display(Image(filename=str(RUN_DIR / "training_curves.png")))

## 4. 디스크에서 모델을 새로 불러오기
메모리의 학습 모델 대신 새 모델에 저장 가중치를 로드합니다. config.json과 best.weights.h5를 함께 보관하세요.

In [ ]:
config = json.loads(CONFIG.read_text(encoding="utf-8"))
vocabulary = config["vocabulary"]
model, encoder, decoder = build_models(config)
model.load_weights(str(WEIGHTS))
reverse_vocabulary = {index: char for char, index in vocabulary.items()}

def chat(question):
    if not question.strip():
        return "질문을 입력해주세요."
    encoded = np.asarray([encode(question, vocabulary, config["max_length"])], dtype="int32")
    state = encoder(encoded, training=False)
    token = np.asarray([[SOS]], dtype="int32")
    answer = []
    for _ in range(config["max_length"]):
        logits, state = decoder([tf.convert_to_tensor(token), state], training=False)
        scores = logits.numpy()[0, -1].copy()
        scores[[PAD, SOS, UNK]] = -np.inf
        index = int(np.argmax(scores))
        if index == EOS:
            break
        answer.append(reverse_vocabulary[index])
        token = np.asarray([[index]], dtype="int32")
    return "".join(answer).strip() or "(모델이 빈 답변을 생성했습니다.)"

print("저장 모델 로드 완료:", WEIGHTS)

## 5. 질문 테스트
아래 질문을 바꾸어 실행하세요. 학습 손실 감소만으로 답변 품질이 보장되지는 않습니다.

In [ ]:
for question in ["안녕", "ESG란 무엇인가요?", "온실가스 배출량은 어떻게 관리하나요?"]:
    print("나:", question)
    print("챗봇:", chat(question))

### 선택: 연속 대화
아래 값을 True로 바꾸면 입력 창이 열립니다. /quit으로 종료합니다. 이전 대화는 기억하지 않습니다.

In [ ]:
INTERACTIVE = False
if INTERACTIVE:
    while True:
        question = input("나: ")
        if question.strip().lower() in {"/quit", "/exit"}:
            break
        print("챗봇:", chat(question))

## 모델 다운로드
다운로드한 ZIP을 풀면 가중치·설정·학습 기록이 있습니다. 새 런타임의 chat 모드에는 best.weights.h5와 config.json을 업로드하세요.

In [ ]:
DOWNLOAD_MODEL = True
if DOWNLOAD_MODEL:
    import shutil
    archive = shutil.make_archive(str(RUN_DIR.parent / (RUN_DIR.name + "_model")), "zip", RUN_DIR)
    print("모델 백업:", archive)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
